# 🧠 Veilcrean — Google Colab Starter

This notebook sets up the **Veilcrean** repository inside Colab and walks you through
the parts that work great in a cloud notebook environment:

- ✅ Install the full brain dependency stack
- ✅ Run the test suite and end-to-end smoke test
- ✅ Offline training on the committed Deriv historical data (no API token needed)
- ✅ Walk-forward backtesting
- ✅ Signal generation (yfinance + public Deriv WebSocket)
- ✅ Optional: fetch *live* historical data from Deriv's public API

> ⚠️ **What cannot run in Colab:** live trading requires the MetaTrader 5 Expert
> Advisor (`mt5_ea/Veilcrean_EA.mq5`) and a ZMQ bridge on a local machine.
> Colab is used for research, training and backtesting — never for execution.

Run the cells in order. Each cell is idempotent, so you can re-run them safely.

In [ ]:
# @title 1. Clone the repository (idempotent)
import os, sys
from pathlib import Path

REPO = Path("/content/Veilcrean")
if not (REPO / ".git").exists():
    print("Cloning Veilcrean...")
    !git clone --depth 1 https://github.com/jvrboy/Veilcrean.git {REPO}
else:
    print("Repository already present — pulling latest...")
    !git -C {REPO} pull --ff-only

os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("Working in:", os.getcwd())
print("Python:", sys.version.split()[0])

In [ ]:
# @title 2. Install dependencies (idempotent, ~2–3 min)
# Colab already ships numpy/pandas/torch, so pip only adds what's missing.
# requirements.brain.txt is the full stack (the root requirements.txt only
# contains the Vercel API deps and is NOT enough for the brain).
!pip install -q -r requirements.brain.txt
!pip install -q -r signals/requirements.txt

# Verify the critical imports
import numpy, pandas, torch, sklearn, zmq, sqlalchemy, matplotlib, plotly
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| torch", torch.__version__)
print("GPU available:", torch.cuda.is_available())

In [ ]:
# @title 3. Verify the repository (tests + smoke test)
!python -m pytest tests/ -q
!python scripts/run_smoke_test.py 2>&1 | tail -5

## 🏋️ Offline training on real Deriv data

The repo ships historical OHLCV data for Deriv's synthetic indices
(`data/historical_deriv/`), so you can train and backtest **with zero API
tokens**. `scripts/train_single.py` walks forward candle-by-candle, generates
scalping signals, simulates each trade and carries learned state
(Q-tables, patterns, TP/SL) between iterations.

In [ ]:
# @title 4. Train one instrument (offline, no token needed)
# Usage: python scripts/train_single.py <SYMBOL> <ITERATIONS> <MAX_SIGNALS>
# Symbols: 1HZ50V, 1HZ75V, 1HZ100V, BOOM500, BOOM900, BOOM1000, CRASH500, CRASH900
!python scripts/train_single.py 1HZ50V 2 60

# Results land in training/output/ as valid JSON
import json
with open("training/output/v3_all_results.json") as f:
    history = json.load(f)
print("Runs recorded:", len(history))

In [ ]:
# @title 5. Walk-forward backtest (offline)
!python scripts/backtest.py --bars 1500 --n-trades 30 2>&1 | tail -8

## 📡 Live data (optional)

These cells need internet access (Colab has it by default). The public Deriv
WebSocket API needs **no token** for historical data; a `DERIV_API_TOKEN` is
only needed for account operations.

In [ ]:
# @title 6. Fetch real candles from Deriv (public API, no token)
import asyncio
import sys
sys.path.insert(0, ".")

from training.deriv_client import DerivClient, TIMEFRAMES

async def demo():
    client = DerivClient()
    try:
        await client.connect()
    except Exception as e:
        print("Could not reach wss://ws.derivws.com:", e)
        print("(Colab usually has internet — retry or skip this cell.)")
        return
    candles = await client.fetch_all_history("R_100", granularity=TIMEFRAMES["5m"], max_batches=2)
    print(f"Fetched {len(candles)} live R_100 5m candles")
    await client.close()

await demo()

In [ ]:
# @title 7. Generate & score signals (yfinance + Deriv)
# Writes signals/latest_signals.json, latest_signals.md and
# performance_report.md. Needs internet for live quotes.
!python -m signals.generate_signals --mode scalp 2>&1 | tail -15 || echo "(signal generation needs internet — see output above)" 

## ✅ What to do next

| Goal | Command (in this notebook or a terminal) |
|---|---|
| Train **all** instruments | `python scripts/train_single.py ALL 5 150` |
| Master DSI trainer | `python MASTER_DSI_TRAINER.py` (needs `DERIV_API_TOKEN`) |
| Full 15-run trainer | `python MASTER_FULL_TRAINER.py` (needs internet) |
| MCP AI-analyst server | `python mcp_server.py` (stdio JSON-RPC) |
| Show performance | `python scripts/show_performance.py` |
| Feature importance | `python scripts/visualize_features.py` |

**For live trading** you still need to run the brain on a machine that can
reach your MT5 terminal: `python run.py` (Windows/Linux — see `docs/MQ5_ZMQ_SETUP.md`).

Happy building! 🚀